# C02–M2–L1 — Set Up a Simple Gradio Interface to Interact with Your Models (Colab edition · Gemini stack)

Converted from the IBM Skills Network lab · [original notebook/files](https://github.com/MohammadWasiq0786/IBM-RAG-and-Agentic-AI/tree/main/02%20Build%20RAG%20Applications/Watsonx.ai%20Chatbot)

**Before you run:** add your free `GOOGLE_API_KEY` ([Google AI Studio](https://aistudio.google.com/apikey)) to Colab 🔑 Secrets and enable notebook access. No GPU needed — a CPU runtime is enough.

**What changed vs. the IBM version**
- Skills Network venv + requirements.txt, the old Groq/HF `get_llm()` setup, the watsonx shim and `llm_factory.py` removed → Cell 1 = minimal Gemini install (+ `gradio`), Cell 2 = universal Gemini init.
- Project files kept as `%%writefile` cells: `gradio_demo.py`, `common_input_types.py`, `simple_llm.py`, `llm_chat.py`.
- `simple_llm.py` / `llm_chat.py`: `WatsonxLLM(model_id, url, project_id, params)` → `ChatGoogleGenerativeAI(model=model_id, **parameters)` built in each script (key from the environment); **Granite 3.3 8B → `gemini-3.8-flash`**, **Mixtral → `gemini-3.5-flash-lite`**; answers via `.invoke(...).text`.
- `launch(server_name="127.0.0.1", server_port=7860)` → `launch(share=True, debug=True)`; `allow_flagging` → `flagging_mode` (also fixes a missing comma in the previous Colab version).
- `input()` in `simple_llm.py` fed with `echo … | python`; Colab exercises at the end (two-textbox app, chat with memory, Gemini model/temperature picker, streaming with `chunk.text`).

Everything else — explanations, code, exercises — is the original lab.

In [ ]:
# Minimal fast install (~5-10 seconds)
!pip install -q -U langchain-google-genai langchain-core langchain-chroma google-genai gradio

### 🔑 Gemini initialization
Run once. Creates `llm` (`gemini-3.8-flash`), `embeddings` (`models/gemini-embedding-2`) and the 1-line `llm_model()` alias used by older lab cells. Needs the `GOOGLE_API_KEY` Colab secret.

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Load API key securely from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Initialize LLM and Embeddings directly (no helper functions or custom shims)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")   # Gemini 3: keep the default temperature (1.0); lower values can cause looping
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# Backward-compatibility alias for older lab cells calling llm_model()
# (.text, not .content: Gemini 3.x can return content as a list of blocks; .text is always a str)
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

print("✅ Google Gemini 3.8 Flash & Gemini Embedding 2 initialized successfully.")

## Step 1 · Your first Gradio app — add two numbers
The lab's `gradio_demo.py`. Running a Gradio script **blocks the cell**: open the `*.gradio.live` link it prints, then press ⏹ (Interrupt) to stop it before running the next cell.

In [ ]:
%%writefile gradio_demo.py
import gradio as gr

def add_numbers(Num1, Num2):
    return Num1 + Num2

# Define the interface
demo = gr.Interface(
    fn=add_numbers, 
    inputs=[gr.Number(), gr.Number()], # Create two numerical input fields where users can enter numbers
    outputs=gr.Number() # Create numerical output fields
)

# Launch the interface
demo.launch(share=True, debug=True)  # Colab: public *.gradio.live link (was server_name="127.0.0.1", server_port=7860)

In [ ]:
!python gradio_demo.py

> 💡 **How to use the ✅ Answer cells:** each exercise is followed by a hidden **✅ Answer** cell (form view, code collapsed). Try the exercise first. To check, tick `run_answer` in the form (or double-click to show the code and set `run_answer = True`) and run the cell. With `run_answer` unticked, the cell stops with *Answer not run* — so **Runtime → Run all** stops at the first answer cell; run the cells below it one by one.

### Exercise 1 · Combine two sentences
Can you create a Gradio application that can combine two input sentences together? Use what you know from the demo and your Python knowledge to create this app. Take your time to complete this exercise. *(lab exercise — the lab's answer is `gradio_concatenation.py`)*

In [ ]:
# Your code here


<details>
    <summary>Click here for a hint</summary>

Start from `gradio_demo.py`: change the function to take two strings and return `sentence1 + " " + sentence2`, and use two `gr.Textbox` inputs and one `gr.Textbox` output.

</details>

In [ ]:
# @title ✅ Answer — Exercise 1: Combine two sentences
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

import gradio as gr   # lab answer: gradio_concatenation.py, run inside the notebook

def combine(a, b):
    return a + " " + b

demo = gr.Interface(
    fn=combine,
    inputs=[gr.Textbox(label="Input 1"), gr.Textbox(label="Input 2")],
    outputs=gr.Textbox(label="Output"),
)
demo.launch(share=True)   # in a notebook launch() returns; stop with demo.close()

# Expected / what you should see: A Gradio app with two textboxes; `Hello` + `world` → `Hello world`.


## Step 2 · Common input types
`gr.Slider`, `gr.Dropdown`, `gr.CheckboxGroup`, `gr.Radio`, multiselect dropdown and `gr.Checkbox`, with clickable `examples`.

In [ ]:
%%writefile common_input_types.py
import gradio as gr

def sentence_builder(quantity, tech_worker_type, countries, place, activity_list, morning):
    return f"""The {quantity} {tech_worker_type}s from {" and ".join(countries)} went to the {place} where they {" and ".join(activity_list)} until the {"morning" if morning else "night"}"""

demo = gr.Interface(
    fn=sentence_builder,
    inputs=[
        gr.Slider(3, 20, value=4, step=1, label="Count", info="Choose between 3 and 20"),
        gr.Dropdown(
            ["Data Scientist", "Software Developer", "Software Engineer"], 
            label="tech_worker_type", 
            info="Will add more tech worker types later!"
        ),
        gr.CheckboxGroup(["Canada", "Japan", "France"], label="Countries", info="Where are they from?"),
        gr.Radio(["office", "restaurant", "meeting room"], label="Location", info="Where did they go?"),
        gr.Dropdown(
            ["partied", "brainstormed", "coded", "fixed bugs"], 
            value=["brainstormed", "fixed bugs"], 
            multiselect=True, 
            label="Activities", 
            info="Which activities did they perform?"
        ),
        gr.Checkbox(label="Morning", info="Did they do it in the morning?"),
    ],
    outputs="text",
    examples=[
        [3, "Software Developer", ["Canada", "Japan"], "restaurant", ["coded", "fixed bugs"], True],
        [4, "Data Scientist", ["Japan"], "office", ["brainstormed", "partied"], False],
        [10, "Software Engineer", ["Canada", "France"], "meeting room", ["brainstormed"], False],
        [8, "Data Scientist", ["France"], "restaurant", ["coded"], True],
    ]
)

demo.launch(share=True, debug=True)  # Colab: public *.gradio.live link (was server_name="127.0.0.1", server_port=7860)

In [ ]:
!python common_input_types.py

## Step 3 · A simple LLM call from the command line
`simple_llm.py` keeps the IBM structure (`model_id` + parameters + one LLM object + `invoke`) but builds `ChatGoogleGenerativeAI` directly; it reads `GOOGLE_API_KEY` from the environment set in Cell 2 (child processes inherit it). The query that the lab types at the prompt is piped in with `echo`.

In [ ]:
%%writefile simple_llm.py
# Import the necessary packages
# Colab (Gemini): ibm_watsonx_ai ModelInference / GenParams / Credentials + langchain_ibm WatsonxLLM
#                 → langchain_google_genai.ChatGoogleGenerativeAI (key from os.environ["GOOGLE_API_KEY"])
from langchain_google_genai import ChatGoogleGenerativeAI

# Specify the model and project settings 
# (make sure the model you wish to use is commented out, and other models are commented)
#model_id = 'gemini-3.5-flash-lite' # was mistralai/mixtral-8x7b-instruct-v01
model_id = 'gemini-3.8-flash' # was ibm/granite-3-3-8b-instruct

# Set the necessary parameters
parameters = {
    "max_output_tokens": 1024,  # was GenParams.MAX_NEW_TOKENS: 256 (headroom for Gemini thinking tokens)
    # "temperature": 0.5,  # was GenParams.TEMPERATURE — Gemini 3: keep the default 1.0 (lower values can loop)
    "thinking_level": "low",  # Gemini 3: thinking tokens count toward max_output_tokens
}

# Wrap up the model (was WatsonxLLM(model_id, url, project_id, params); no project_id needed)
watsonx_llm = ChatGoogleGenerativeAI(model=model_id, **parameters)

# Get the query from the user input
query = input("Please enter your query: ")

# Print the generated response (.text = plain string of the chat answer)
print(watsonx_llm.invoke(query).text)

# How to be a good data scientist?

In [ ]:
!echo "How to be a good data scientist?" | python simple_llm.py

## Step 4 · Put the LLM behind a Gradio interface — the Watsonx.ai Chatbot
`llm_chat.py` wraps `watsonx_llm.invoke(...).text` in `gr.Interface(fn=generate_response, …)`. Try the lab's question *How to be a good data scientist?* and a few of your own; swap the commented `model_id` lines to compare models.

In [ ]:
%%writefile llm_chat.py
# Import necessary packages
# Colab (Gemini): ibm_watsonx_ai ModelInference / GenParams / Credentials + langchain_ibm WatsonxLLM
#                 → langchain_google_genai.ChatGoogleGenerativeAI (key from os.environ["GOOGLE_API_KEY"])
from langchain_google_genai import ChatGoogleGenerativeAI
import gradio as gr

# Model and project settings
model_id = 'gemini-3.8-flash' # was ibm/granite-3-3-8b-instruct (active in the lab)
#model_id = 'gemini-3.5-flash-lite' # was mistralai/mistral-small-3-1-24b-instruct-2503 (the lab's comment calls it "Mixtral 8x7B")

# Set necessary parameters
parameters = {
    # "max_output_tokens": 256,  # Specify the max tokens you want to generate
    "max_output_tokens": 1024,  # Specifying the max tokens you want to generate
    # "temperature": 0.5,  # was GenParams.TEMPERATURE — Gemini 3: keep the default 1.0
    "thinking_level": "low",  # Gemini 3: thinking tokens count toward max_output_tokens
}

# Wrap up the model (was WatsonxLLM(model_id, url, project_id, params))
watsonx_llm = ChatGoogleGenerativeAI(model=model_id, **parameters)

# Function to generate a response from the model
def generate_response(prompt_txt):
    generated_response = watsonx_llm.invoke(prompt_txt).text
    return generated_response

# Create Gradio interface
chat_application = gr.Interface(
    fn=generate_response,
    flagging_mode="never",  # Gradio 5: was allow_flagging="never"
    inputs=gr.Textbox(label="Input", lines=2, placeholder="Type your question here..."),
    outputs=gr.Textbox(label="Output"),
    title="Watsonx.ai Chatbot",
    description="Ask any question and the chatbot will try to answer."
)

# Launch the app
chat_application.launch(share=True, debug=True)  # Colab: public link (was server_name="127.0.0.1", server_port=7860)

In [ ]:
!python llm_chat.py

### Exercise 2 · Incomplete answers
You might observe that responses from the LLM are occasionally incomplete. How can you identify the cause of this issue? Also, can you modify the code to enable the model to generate more content? *(lab exercise)*

In [ ]:
# Your code here


<details>
    <summary>Click here for a hint</summary>

The lab's answer: increase `max_new_tokens` (Gemini: `max_output_tokens`). On Gemini 3, check `response.response_metadata["finish_reason"]` — `MAX_TOKENS` means the budget ran out, and **thinking tokens count toward it**, so also set `thinking_level="low"`.

</details>

In [ ]:
# @title ✅ Answer — Exercise 2: Incomplete answers
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

from langchain_google_genai import ChatGoogleGenerativeAI
import re

q = "Write a detailed guide on how to be a good data scientist."

# 1) Find the cause: finish_reason MAX_TOKENS = cut off. Gemini 3 counts thinking tokens in the budget.
short = ChatGoogleGenerativeAI(model="gemini-3.8-flash", max_output_tokens=256)   # the lab's 256
r = short.invoke(q)
print("256 tokens  → finish_reason:", r.response_metadata.get("finish_reason"),
      "| thinking tokens:", (r.usage_metadata or {}).get("output_token_details", {}).get("reasoning"),
      "| answer chars:", len(r.text))

# 2) Fix: a bigger budget (and little thinking)
long = ChatGoogleGenerativeAI(model="gemini-3.8-flash", max_output_tokens=4096, thinking_level="low")
r2 = long.invoke(q)
print("4096 tokens → finish_reason:", r2.response_metadata.get("finish_reason"), "| answer chars:", len(r2.text))

# 3) Apply it to the app script and run it again (!python llm_chat.py)
s = open("llm_chat.py").read()
s = re.sub(r'"max_output_tokens": \d+,(?=  # Specifying)', '"max_output_tokens": 4096,', s)
open("llm_chat.py", "w").write(s)
print([l.strip() for l in s.splitlines() if l.strip().startswith('"max_output_tokens"')])

# Expected / what you should see: With 256 tokens: `finish_reason: MAX_TOKENS`, many thinking tokens, a short or empty answer. With 4096 + `thinking_level="low"`: `STOP` and a full answer. The last line shows `"max_output_tokens": 4096` in `llm_chat.py`.


---
# 🧪 Colab practice exercises
These run **inside the notebook** (no script), using `llm` from Cell 2. In a notebook, `launch()` returns immediately, so you can keep working while the app is up; call `demo.close()` to stop it.

### C1 · Combine two texts (reference-port exercise)
Build an interface with two `gr.Textbox` inputs that returns them joined with a space.

In [ ]:
import gradio as gr
def combine(a, b):
    return a + " " + b   # TODO: also return the word count in a second output
demo = gr.Interface(fn=combine, inputs=[gr.Textbox(label="Input 1"), gr.Textbox(label="Input 2")],
                    outputs=gr.Textbox(label="Output"))
demo.launch(share=True)

In [ ]:
# @title ✅ Answer — C1 · Combine two texts
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

import gradio as gr

def combine(a, b):
    text = a + " " + b
    return text, len(text.split())          # second output = word count

demo = gr.Interface(fn=combine,
                    inputs=[gr.Textbox(label="Input 1"), gr.Textbox(label="Input 2")],
                    outputs=[gr.Textbox(label="Output"), gr.Number(label="Word count")])
print(combine("Hello there", "general Kenobi"))   # ('Hello there general Kenobi', 4)
demo.launch(share=True)

# Expected / what you should see: `('Hello there general Kenobi', 4)` and an app with *Output* + *Word count*.


### C2 · Chat UI with memory
Use `gr.ChatInterface` and pass the history to the model so follow-up questions work.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

def _text(content):                            # Gradio 6 may send content as a list of parts
    if isinstance(content, list):
        return " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    return str(content)

def chat(message, history):
    msgs = [SystemMessage("You are a concise, friendly assistant.")]
    for turn in history:                       # "messages" format: {"role", "content"}
        msgs.append(HumanMessage(_text(turn["content"])) if turn["role"] == "user" else AIMessage(_text(turn["content"])))
    msgs.append(HumanMessage(_text(message)))
    return llm.invoke(msgs).text
chat_demo = gr.ChatInterface(chat, title="Colab LLM Chat")   # messages format is the default in Gradio 5.x/6
chat_demo.launch(share=True)
# TODO: ask "My name is Sam" then "What is my name?" — then remove the history loop and compare.

In [ ]:
# @title ✅ Answer — C2 · Chat UI with memory
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

from langchain_core.messages import HumanMessage, SystemMessage

def chat_no_memory(message, history):         # history ignored
    return llm.invoke([SystemMessage("You are a concise, friendly assistant."), HumanMessage(_text(message))]).text

# Same 2-turn test without the UI (history in Gradio "messages" format)
h = [{"role": "user", "content": "My name is Sam"},
     {"role": "assistant", "content": chat("My name is Sam", [])}]
print("with memory   :", chat("What is my name?", h))
print("without memory:", chat_no_memory("What is my name?", h))

gr.ChatInterface(chat_no_memory, title="No-memory chat").launch(share=True)

# Expected / what you should see: *with memory* → "Your name is Sam"; *without memory* → the model says it doesn't know your name.


### C3 · Model and temperature picker
Add a `gr.Dropdown` of models and a `gr.Slider` for temperature to the Step 4 chatbot.

In [ ]:
MODELS = ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-2.5-flash"]
def answer(prompt, model_name, temperature):
    chat = ChatGoogleGenerativeAI(model=model_name, temperature=temperature)
    return chat.invoke(prompt).text
picker = gr.Interface(answer,
    inputs=[gr.Textbox(label="Input", lines=2), gr.Dropdown(MODELS, value="gemini-3.8-flash", label="Model"),
            gr.Slider(0, 1, value=0.5, step=0.1, label="Temperature")],
    outputs=gr.Textbox(label="Output"), title="Model playground")
picker.launch(share=True)
# TODO: add a "max output tokens" slider and show the response time in a second output.

In [ ]:
# @title ✅ Answer — C3 · Model and temperature picker
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

import time
MODELS = ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash"]

def answer(prompt, model_name, temperature, max_tokens):
    chat = ChatGoogleGenerativeAI(model=model_name, temperature=temperature, max_output_tokens=int(max_tokens))
    t = time.time()
    try:
        text = chat.invoke(prompt).text
    except Exception as e:                      # 2.5 access limit / free-tier 429
        text = f"⚠️ {type(e).__name__}: {str(e)[:200]}"
    return text, f"{time.time() - t:.2f} s"

picker = gr.Interface(answer,
    inputs=[gr.Textbox(label="Input", lines=2),
            gr.Dropdown(MODELS, value="gemini-3.8-flash", label="Model"),
            gr.Slider(0, 2, value=1.0, step=0.1, label="Temperature (Gemini 3: keep 1.0)"),
            gr.Slider(64, 4096, value=1024, step=64, label="Max output tokens (includes thinking)")],
    outputs=[gr.Textbox(label="Output"), gr.Textbox(label="Response time")],
    title="Model playground")
print(answer("Say hi in 3 words", "gemini-3.8-flash", 1.0, 1024))
picker.launch(share=True)

# Expected / what you should see: A tuple (answer, `'0.8 s'`-style time); the app has 4 inputs + 2 outputs. flash-lite ignores the temperature (warning).


### C4 · Streaming answers
Gradio streams when the function is a generator. Stream tokens from `llm.stream(...)`.

In [ ]:
def stream_answer(prompt):
    text = ""
    for chunk in llm.stream(prompt):
        text += chunk.text   # .text: Gemini chunks can carry content blocks
        yield text
gr.Interface(stream_answer, gr.Textbox(label="Input"), gr.Textbox(label="Output"), title="Streaming chatbot").launch(share=True)
# TODO: rebuild this with gr.Blocks: a Textbox, a Button and a Markdown output side by side.

In [ ]:
# @title ✅ Answer — C4 · Streaming answers
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

with gr.Blocks(title="Streaming chatbot (Blocks)") as blocks_demo:
    with gr.Row():
        with gr.Column():
            box = gr.Textbox(label="Input", lines=3)
            btn = gr.Button("Ask", variant="primary")
        out = gr.Markdown()
    btn.click(stream_answer, inputs=box, outputs=out)   # generator → streams into the Markdown
    box.submit(stream_answer, inputs=box, outputs=out)

print("chunks:", len(list(stream_answer("Explain RAG in one sentence"))))
blocks_demo.launch(share=True)

# Expected / what you should see: `chunks: N` (N ≥ 1), then a Blocks app: textbox + button on the left, streaming Markdown on the right.
